In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'tensorflow', 'keras', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'keras': 'keras==3.14.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'tensorflow.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Backpropagation

In that section we trained an MLP without writing a line
of calculus: the framework supplied the gradients. Every step of minibatch
stochastic gradient descent needed the derivative of the loss with respect to
each of the network's 203,530 parameters, and automatic differentiation
supplied all of them for a small multiple of the cost of the forward pass. How
can one pass backward through a network compute so many derivatives at once?

that section gave the answer in general terms: the framework records
the forward pass as a computational graph and applies the chain rule backward
through it, one local rule per operation, each reading values that the forward
pass saved. There the backward pass through a network remained a black box,
which this section opens. Little of the machinery is new. What is new is the
setting: the parameters are matrices, a nonlinearity sits between the layers,
and the gradient has to pass *through* one layer to reach the one below. An
MLP needs only three local rules, for a matrix product, for adding a bias, and
for an elementwise activation function, and applying them from the loss back
to the first layer is backpropagation.

Three practical questions depend on the details. How much memory do the saved
values take, and why does it grow with the minibatch? Why does a ReLU unit
that is inactive for every example stop learning? And why do the gradients of
a deep network vanish or explode, the subject of the next section? After a
brief reminder on a graph of three numbers, we derive the three rules in
matrix form, read the result as an error sent backward through the network,
check it against automatic differentiation, and count what it costs in memory
and time.

In [ ]:
import math
import numpy as np
import tensorflow as tf
from d2l import tensorflow as d2l

## Computational Graphs

### A Small Graph

The graph that we traced in that section had vectors at its nodes.
As a reminder of how a backward pass works, here are the two most common
local rules in their barest form, on three numbers: $e = (a + b)\,c$ at
$a = 2$, $b = 1$, and $c = -3$. The forward pass computes the intermediate
value $u = a + b = 3$ and then $e = u\,c = -9$. The backward pass starts
from $\partial e / \partial e = 1$ and multiplies by local derivatives on
its way to the inputs. The multiplication has
$\partial e / \partial u = c = -3$ and $\partial e / \partial c = u = 3$,
the addition has
$\partial u / \partial a = \partial u / \partial b = 1$, and the chain rule
gives

$$
\frac{\partial e}{\partial a} = \frac{\partial e}{\partial u}\,\frac{\partial u}{\partial a} = -3, \qquad
\frac{\partial e}{\partial b} = \frac{\partial e}{\partial u}\,\frac{\partial u}{\partial b} = -3, \qquad
\frac{\partial e}{\partial c} = u = 3.
$$

An addition copies the incoming gradient to each of its inputs, and a
multiplication scales it by the value of the *other* input.
the figure shows both passes.

![The graph of $e = (a + b)\,c$ at $a = 2$, $b = 1$, $c = -3$. Each node shows its value (black) and the derivative of $e$ with respect to it (red). The forward pass (blue) runs from left to right; the backward pass (red) starts from 1 at $e$ and multiplies by the local derivative written on each red arrow.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-mlp-backprop-small.svg)

Both habits of a backward pass are already visible here. It reads values
that the forward pass computed: the multiplication needed both of its inputs,
$c$ for the gradient of $u$ and $u$ for that of $c$, while the addition
needed none. And it runs in the reverse order of the forward pass, here $e$
first, then $u$ and $c$, then $a$ and $b$, since a gradient is ready only
once all the nodes that use its value have passed theirs back.

### The Graph of an MLP

Now to the network, the MLP of that section. A minibatch of
$n$ examples forms the rows of $\mathbf{X} \in \mathbb{R}^{n \times d}$. The
hidden layer has weights $\mathbf{W}^{(1)} \in \mathbb{R}^{d \times h}$,
biases $\mathbf{b}^{(1)} \in \mathbb{R}^{h}$, and activation function
$\phi$, and the output layer has weights
$\mathbf{W}^{(2)} \in \mathbb{R}^{h \times q}$ and biases
$\mathbf{b}^{(2)} \in \mathbb{R}^{q}$. The backward pass needs a name for the
input of $\phi$, the pre-activation $\mathbf{Z}$, so we write the forward
pass in three steps:

$$
\begin{aligned}
\mathbf{Z} &= \mathbf{X}\mathbf{W}^{(1)} + \mathbf{b}^{(1)}, \\
\mathbf{H} &= \phi(\mathbf{Z}), \\
\mathbf{O} &= \mathbf{H}\mathbf{W}^{(2)} + \mathbf{b}^{(2)}.
\end{aligned}
$$

The first two lines are the hidden layer, and the third computes the logits.
The loss is the cross-entropy of that section. Row $i$ of
$\hat{\mathbf{Y}} = \mathrm{softmax}(\mathbf{O})$ holds the predicted class
probabilities of example $i$, its loss $l_i = -\log \hat{y}_{i, y_i}$ is the
negative log-probability of its true class $y_i$, and $L$ is the average of
$l_i$ over the $n$ examples. We also add weight decay
(that section), which training usually includes and which makes
each weight matrix feed two nodes, so that we see how gradients from several
paths combine. Its penalty is

$$s = \frac{\lambda}{2}\left(\|\mathbf{W}^{(1)}\|_\textrm{F}^2 + \|\mathbf{W}^{(2)}\|_\textrm{F}^2\right),$$

where the squared Frobenius norm $\|\mathbf{W}\|_\textrm{F}^2$ is the sum of
the squares of all entries of $\mathbf{W}$. Training minimizes the
*objective* $J = L + s$.

the figure draws this computation as a graph. Its bottom row is
the network itself: the input $\mathbf{X}$, the two values of the hidden
layer, $\mathbf{Z}$ and $\mathbf{H}$, the logits $\mathbf{O}$, and the loss
$L$. Each node is labeled with its shape for the MLP of
that section, which has $d = 784$ inputs, $h = 256$
hidden units, and $q = 10$ classes. The weight matrices sit above the layers
that use them, and each of them feeds two nodes: its layer and the penalty
$s$.

![The forward pass of an MLP with one hidden layer and weight decay. Nodes are values, labeled with their shapes for $d = 784$, $h = 256$, and $q = 10$; parameters are shaded. The biases are omitted: they enter $\mathbf{Z}$ and $\mathbf{O}$ like the weights, but not the penalty $s$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-mlp-forward-graph.svg)

## Backpropagation through an MLP

### Local Rules for Matrices

The backward pass of the MLP computes the gradient of the scalar $J$ with
respect to matrices. We write $\partial J / \partial \mathbf{W}$ for the
matrix of partial derivatives $\partial J / \partial w_{rj}$, which has the
same shape as $\mathbf{W}$. Each node of the figure needs a rule
that turns the gradient with respect to its output into gradients with
respect to its inputs.

Let us start with the matrix product, the only one of the three rules that
takes some care. In that section a matrix–vector product passed the
gradient back to its vector through $\mathbf{X}^\top$. Now both factors are
matrices, and we need the gradient with respect to each. Consider
$\mathbf{A} = \mathbf{X}\mathbf{W}$ with
$\mathbf{X} \in \mathbb{R}^{n \times d}$ and
$\mathbf{W} \in \mathbb{R}^{d \times h}$, inside a larger computation whose
gradient $\mathbf{G} = \partial J / \partial \mathbf{A}$ is known. Entry
$a_{ij} = \sum_r x_{ir} w_{rj}$ depends on $w_{rj}$ with derivative $x_{ir}$,
and $w_{rj}$ enters every entry of column $j$ of $\mathbf{A}$. The chain rule
for many variables (that section) adds up the contributions of
all these paths: $\partial J / \partial w_{rj} = \sum_i g_{ij} x_{ir}$,
which is entry $(r, j)$ of $\mathbf{X}^\top \mathbf{G}$. In the same way,
$x_{ir}$ enters every entry of row $i$, and
$\partial J / \partial x_{ir} = \sum_j g_{ij} w_{rj}$. In matrix form,

$$
\frac{\partial J}{\partial \mathbf{W}} = \mathbf{X}^\top \mathbf{G}, \qquad
\frac{\partial J}{\partial \mathbf{X}} = \mathbf{G}\,\mathbf{W}^\top.
$$

Shapes give a quick check: $\mathbf{X}^\top \mathbf{G}$ is $d \times h$, like
$\mathbf{W}$, and $\mathbf{G}\mathbf{W}^\top$ is $n \times d$, like
$\mathbf{X}$, and when $n$, $d$, and $h$ differ, no other product of the two
factors has the right shape. Row $i$ of $\mathbf{X}$ is $\mathbf{x}_i^\top$,
the transposed input $\mathbf{x}_i \in \mathbb{R}^d$ of example $i$, and row
$i$ of $\mathbf{G}$ is $\mathbf{g}_i^\top$ with
$\mathbf{g}_i \in \mathbb{R}^h$. The weight gradient
$\mathbf{X}^\top\mathbf{G} = \sum_i \mathbf{x}_i \mathbf{g}_i^\top$ is
therefore a sum of $n$ outer products, one per example. The rule is the
multiplication node of the small graph in matrix form. The gradient of each
factor involves the *other* factor, so the forward pass must keep $\mathbf{X}$
for the gradient of $\mathbf{W}$. The second half of the rule,
$\mathbf{G}\mathbf{W}^\top$, is what hands the gradient on to the layer below.

The other two operations are simpler. Adding the bias,
$\mathbf{Z} = \mathbf{A} + \mathbf{b}$, adds $b_j$ to every entry of column
$j$. Like the addition of the small graph, it passes the incoming gradient
$\mathbf{G} = \partial J / \partial \mathbf{Z}$ through unchanged,
$\partial J / \partial \mathbf{A} = \mathbf{G}$, and the gradient of the
bias is the sum of the rows of $\mathbf{G}$, one row per example. The
activation $\mathbf{H} = \phi(\mathbf{Z})$ acts on each entry separately, so
entry $(i, j)$ of $\mathbf{H}$ depends on entry $(i, j)$ of $\mathbf{Z}$
alone, and

$$\frac{\partial J}{\partial \mathbf{Z}} = \frac{\partial J}{\partial \mathbf{H}} \odot \phi'(\mathbf{Z}),$$

where $\odot$ multiplies matrices entry by entry. Each entry of the gradient
is scaled by the slope of $\phi$ at the corresponding pre-activation. This
rule needs $\mathbf{Z}$. For the ReLU, $\phi'(z)$ is 1 for $z > 0$ and 0
otherwise, so the rule passes the gradient of every active unit and blocks
that of every inactive one. The ReLU can read the slope off its output, since
$[\mathbf{H}]_{ij} > 0$ exactly when $[\mathbf{Z}]_{ij} > 0$. All three rules
are special cases of the vector–Jacobian products of that section;
the appendix derives them for general tensor operations in
that section.

### The Backward Pass

The three rules are all that the MLP needs. The backward pass starts at the
objective with $\partial J / \partial J = 1$. Since $J = L + s$, both $L$ and
$s$ receive the gradient 1. For the cross-entropy, that section
showed that the gradient of the loss of one example with respect to its logits
is $\hat{\mathbf{y}} - \mathbf{y}$, the predicted probabilities minus the
one-hot label. With the one-hot labels as the rows of
$\mathbf{Y} \in \{0, 1\}^{n \times q}$ and the average over $n$ examples, this
gives the first line below. Each further line applies one of the local rules,
following the arrows of the figure backward from $J$, right to left
along the bottom row:

$$
\begin{aligned}
\frac{\partial J}{\partial \mathbf{O}} &= \frac{1}{n}\bigl(\hat{\mathbf{Y}} - \mathbf{Y}\bigr), \\
\frac{\partial J}{\partial \mathbf{W}^{(2)}} &= \mathbf{H}^\top \frac{\partial J}{\partial \mathbf{O}} + \lambda \mathbf{W}^{(2)}, \qquad
\frac{\partial J}{\partial \mathbf{b}^{(2)}} = \Bigl(\frac{\partial J}{\partial \mathbf{O}}\Bigr)^\top \mathbf{1}, \\
\frac{\partial J}{\partial \mathbf{H}} &= \frac{\partial J}{\partial \mathbf{O}}\, \mathbf{W}^{(2)\top}, \\
\frac{\partial J}{\partial \mathbf{Z}} &= \frac{\partial J}{\partial \mathbf{H}} \odot \phi'(\mathbf{Z}), \\
\frac{\partial J}{\partial \mathbf{W}^{(1)}} &= \mathbf{X}^\top \frac{\partial J}{\partial \mathbf{Z}} + \lambda \mathbf{W}^{(1)}, \qquad
\frac{\partial J}{\partial \mathbf{b}^{(1)}} = \Bigl(\frac{\partial J}{\partial \mathbf{Z}}\Bigr)^\top \mathbf{1}.
\end{aligned}
$$

Here $\mathbf{1} \in \mathbb{R}^n$ is the vector of all ones, so
$\mathbf{G}^\top \mathbf{1}$ sums the rows of $\mathbf{G}$ into a column
vector, of the shape of the bias. Read from top to bottom,
the equation gives the order of the backward pass: the gradients
reach the output layer first and the hidden layer last, and no line computes
$\partial J / \partial \mathbf{X}$, because nothing below the input needs it.
It also tells us what the forward pass must keep, namely the values that the
lines read: $\hat{\mathbf{Y}}$, $\mathbf{H}$, $\mathbf{W}^{(2)}$,
$\mathbf{Z}$, and $\mathbf{X}$.

The display also has a meaning, and its third line is where the method gets
its name. The first line is the error at the outputs: the predicted
probabilities minus the labels. The third line sends this error back through
the same weights that carried the hidden activations forward, so that hidden
unit $j$ receives the errors of the $q$ outputs, each multiplied by the weight
from unit $j$ to that output. A hidden unit has no label of its own. What it
learns from is this message: how wrong the outputs that it feeds are, and in
which direction. The fourth line lets the message through only where the slope
of the unit's activation is not zero.

The weights receive two terms. $\mathbf{W}^{(2)}$ reaches $J$ along two paths
in the figure, through $\mathbf{O}$ and $L$ and through the penalty
$s$, and the chain rule adds the contributions of all paths, as it did for the
product $z = ux$ in that section:
$\mathbf{H}^\top \partial J / \partial \mathbf{O}$ from the first and
$\lambda \mathbf{W}^{(2)}$ from the second. A hand-written backward pass that
assigns the second contribution instead of adding it, `=` where `+=` belongs,
computes a wrong gradient without any error message.

Nothing here is special to a single hidden layer. Let layer $k$ of $K$ compute
$\mathbf{Z}^{(k)} = \mathbf{H}^{(k-1)}\mathbf{W}^{(k)} + \mathbf{b}^{(k)}$,
with $\mathbf{H}^{(0)} = \mathbf{X}$,
$\mathbf{H}^{(k)} = \phi(\mathbf{Z}^{(k)})$ in the hidden layers, and the
logits $\mathbf{O} = \mathbf{Z}^{(K)}$. The backward pass starts as before,
from
$\partial J / \partial \mathbf{Z}^{(K)} = \frac{1}{n}(\hat{\mathbf{Y}} - \mathbf{Y})$,
and repeats one step from the last hidden layer down to the first:

$$
\frac{\partial J}{\partial \mathbf{Z}^{(k)}} = \Bigl(\frac{\partial J}{\partial \mathbf{Z}^{(k+1)}}\, \mathbf{W}^{(k+1)\top}\Bigr) \odot \phi'\bigl(\mathbf{Z}^{(k)}\bigr), \qquad
\frac{\partial J}{\partial \mathbf{W}^{(k)}} = \mathbf{H}^{(k-1)\top} \frac{\partial J}{\partial \mathbf{Z}^{(k)}} + \lambda \mathbf{W}^{(k)}.
$$

The first equation merges lines 3 and 4 of the equation; the
second is the rule for the weights, which holds for the last layer as well. A
layer need not know how many others there are. It receives the error of the
layer above, takes its own gradients from it, and passes it on, and on the way
down the error is multiplied by one transposed weight matrix and one matrix of
slopes per layer.

### A Worked Example

We now run the equation by hand on a network small enough to
check every number: $d = h = 2$ inputs and hidden units, $q = 1$ output, a
single example, ReLU activation, zero biases, and no weight decay. The
output is a single number $o$, and the loss is the squared error
$L = \frac{1}{2}(o - y)^2$. We take

$$
\mathbf{X} = \begin{bmatrix} 1 & 2 \end{bmatrix},\quad
\mathbf{W}^{(1)} = \begin{bmatrix} \phantom{-}1 & 0 \\ -1 & 1 \end{bmatrix},\quad
\mathbf{W}^{(2)} = \begin{bmatrix} \phantom{-}2 \\ -1 \end{bmatrix},\quad
y = 0.
$$

We treat the single example as a minibatch with $n = 1$, so $\mathbf{Z}$,
$\mathbf{H}$, and their gradients are $1 \times 2$ matrices, rows of two
numbers like $\mathbf{X}$. The forward pass gives
$\mathbf{Z} = \mathbf{X}\mathbf{W}^{(1)} = \begin{bmatrix} -1 & 2 \end{bmatrix}$ and
$\mathbf{H} = \begin{bmatrix} 0 & 2 \end{bmatrix}$: the first hidden unit is
inactive. Then
$o = \mathbf{H}\mathbf{W}^{(2)} = -2$ and $L = \frac{1}{2}(-2)^2 = 2$. For
the squared loss, only the first line of the equation changes:
$\partial L / \partial o = o - y = -2$. The other lines give

$$
\begin{aligned}
\frac{\partial L}{\partial \mathbf{W}^{(2)}} &= \mathbf{H}^\top \frac{\partial L}{\partial o} = \begin{bmatrix} 0 \\ 2 \end{bmatrix}(-2) = \begin{bmatrix} \phantom{-}0 \\ -4 \end{bmatrix}, &
\frac{\partial L}{\partial \mathbf{H}} &= \frac{\partial L}{\partial o}\,\mathbf{W}^{(2)\top} = \begin{bmatrix} -4 & 2 \end{bmatrix}, \\
\frac{\partial L}{\partial \mathbf{Z}} &= \begin{bmatrix} -4 & 2 \end{bmatrix} \odot \begin{bmatrix} 0 & 1 \end{bmatrix} = \begin{bmatrix} 0 & 2 \end{bmatrix}, &
\frac{\partial L}{\partial \mathbf{W}^{(1)}} &= \mathbf{X}^\top \frac{\partial L}{\partial \mathbf{Z}} = \begin{bmatrix} 1 \\ 2 \end{bmatrix}\begin{bmatrix} 0 & 2 \end{bmatrix} = \begin{bmatrix} 0 & 2 \\ 0 & 4 \end{bmatrix}.
\end{aligned}
$$

Column $j$ of $\mathbf{W}^{(1)}$ holds the weights into hidden unit $j$, and
the first column of $\partial L / \partial \mathbf{W}^{(1)}$ is zero: the ReLU
passed no gradient back to the inactive unit, so this example does not change
the weights into it. A unit that is inactive for every training example
receives no gradient from any of them: this is the dead unit of
that section, seen in its gradient. The bias gradients follow from the
rows that we already have. For a single example, the sum over rows is the row
itself, so $\partial L / \partial \mathbf{b}^{(1)} = (0, 2)$ holds the entries
of $\partial L / \partial \mathbf{Z}$, and
$\partial L / \partial \mathbf{b}^{(2)} = \partial L / \partial o = -2$.
the figure shows all the values and gradients in the
graph.

![The worked example as a graph. Each node shows its value (black) and the gradient of $L$ with respect to it (red); the biases are zero and omitted. The forward pass (blue) runs left to right and down into each layer; the backward pass (red) applies the rule written on each red arrow. The inactive first hidden unit, $z_1 = -1$, receives no gradient, which zeros the first column of $\partial L / \partial \mathbf{W}^{(1)}$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-mlp-backprop-graph.svg)

### Checking the Backward Pass on a Minibatch

The worked example was chosen to be easy: a single example, the squared loss,
no weight decay. To test every line of the equation, we implement
it for the full MLP and compare the result with automatic differentiation. The
function `mlp_backward` runs the forward pass the equation,
keeping the values that the backward pass reads, and then the five lines of
the equation, with ReLU as $\phi$.

In [ ]:
def mlp_backward(X, Y, W1, b1, W2, b2, lambd):
    """Gradients of J = cross-entropy + weight decay, by backprop."""
    Z = X @ W1 + b1  # forward pass, keeping X, Z, H, and Y_hat
    H = tf.nn.relu(Z)
    Y_hat = tf.nn.softmax(H @ W2 + b2, axis=1)
    G_O = (Y_hat - Y) / X.shape[0]  # dJ/dO
    dW2 = tf.matmul(H, G_O, transpose_a=True) + lambd * W2
    G_Z = tf.matmul(G_O, W2, transpose_b=True) * tf.cast(Z > 0, Z.dtype)
    dW1 = tf.matmul(X, G_Z, transpose_a=True) + lambd * W1
    return dW1, tf.reduce_sum(G_Z, 0), dW2, tf.reduce_sum(G_O, 0)

We draw a minibatch of 256 random inputs and labels, initialize the
parameters as in that section, with weights from
$\mathcal{N}(0, 0.01^2)$ and zero biases, and set $\lambda = 10^{-3}$.
Automatic differentiation differentiates the framework's own cross-entropy
loss. As is usual for gradient checks, we compute in 64-bit floating point,
whose rounding error, about $10^{-16}$ relative to each result, is far
smaller than the error that a wrong line would cause. For each parameter we
print the largest difference between the two gradients, relative to the
largest entry of the gradient.

In [ ]:
n, d, h, q, lambd = 256, 784, 256, 10, 1e-3
f64 = tf.float64  # 64-bit floating point throughout the check
X = tf.random.normal((n, d), dtype=f64)
y = tf.random.uniform((n,), 0, q, dtype=tf.int32)
params = [tf.Variable(tf.random.normal((d, h), stddev=0.01, dtype=f64)),
          tf.Variable(tf.zeros(h, f64)),
          tf.Variable(tf.random.normal((h, q), stddev=0.01, dtype=f64)),
          tf.Variable(tf.zeros(q, f64))]
W1, b1, W2, b2 = params
with tf.GradientTape() as tape:
    O = tf.nn.relu(X @ W1 + b1) @ W2 + b2
    J = (tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(y, O))
         + lambd / 2 * (tf.reduce_sum(W1 ** 2) + tf.reduce_sum(W2 ** 2)))
auto = tape.gradient(J, params)  # automatic differentiation
grads = mlp_backward(X, tf.one_hot(y, q, dtype=f64), *params, lambd)
for name, a, g in zip(['W1', 'b1', 'W2', 'b2'], auto, grads):
    rel = tf.reduce_max(tf.abs(a - g)) / tf.reduce_max(tf.abs(a))
    print(f'{name}: relative difference {float(rel):.1e}')

All relative differences are below $10^{-14}$, within a few rounding errors
of 64-bit floating point: for this network, the equation is the
computation that automatic differentiation performs.

## Memory and Computation

### Stored Values

The backward pass reads values that the forward pass computed, and
that section named this as the reason why training needs more memory
than prediction. For the MLP of that section we can say
how much more. The lines of the equation read $\mathbf{X}$,
$\mathbf{H}$, and $\hat{\mathbf{Y}}$, and with the ReLU, $\mathbf{H}$ also
determines $\phi'(\mathbf{Z})$, so $\mathbf{Z}$ need not be kept. A minibatch
of $n = 256$ examples has

$$256 \times 784 + 256 \times 256 + 256 \times 10 = 200{,}704 + 65{,}536 + 2{,}560 = 268{,}800$$

such numbers, which take 1.1 MB in 32-bit floating point at 4 bytes each. The
network has only 203,530 parameters, 0.8 MB. Even for this small network,
then, the values saved for one minibatch outweigh the model. Unlike the
parameters, they grow in proportion to the minibatch size $n$, and every
additional hidden layer of width 256 adds another $256 \times 256$ matrix per
minibatch. Prediction stores nothing for later: it can discard each layer's
input as soon as the layer's output is computed. The largest minibatch that
fits on a GPU is therefore often set by the stored values and not by the
parameters.

### Computation

What about time? Almost all of the work lies in the matrix products. For each
example, the forward pass of the MLP computes
$784 \times 256 + 256 \times 10 = 203{,}264$ multiply-adds. Each forward
matrix product turns into two backward products of the same size,
the equation, one for the weights and one for the input, except
that the first layer needs no gradient for its input $\mathbf{X}$. The
backward pass therefore computes
$784 \times 256 + 2 \times 256 \times 10 = 205{,}824$ multiply-adds per
example. A training step costs about two forward passes for this network and
about three for a network whose first layer is small, since then nearly every
product appears twice in the backward pass.

This settles the question we started with. Finite differences would need one
forward pass per parameter, some two hundred thousand per training step. The
backward pass computes $\partial J / \partial \mathbf{O}$ and
$\partial J / \partial \mathbf{Z}$ once, and the derivative of the loss with
respect to every parameter is read off these two matrices by a single
product or sum: the work that the derivatives have in common is done once.
This is the advantage of reverse mode (that section), and here we
can point to the shared quantities: they are the errors of the layers. In a
deep network the error passes through many layers, and the next section asks
what a long product of transposed weight matrices and slopes does to its
size.

## Summary

Backpropagation is the reverse mode of that section applied to a
network. Three local rules suffice for an MLP: an addition passes the
incoming gradient through, a matrix product $\mathbf{X}\mathbf{W}$ passes
back $\mathbf{X}^\top\mathbf{G}$ to the weights and
$\mathbf{G}\mathbf{W}^\top$ to its input, and an elementwise activation
multiplies by its slope $\phi'(\mathbf{Z})$, which for the ReLU is zero at
every inactive unit. A value that feeds several nodes adds up the gradients
that they pass back. Each layer receives the error of the layer above, sent
back through the weights between them, and every parameter gradient is read
off these shared per-layer errors, which is why one pass suffices. The rules
read values from the forward pass, which training must store: for the MLP of
that section, 1.1 MB per minibatch of 256 examples, more
than its parameters, and growing with the minibatch size and the depth.

## Exercises

1. **Jacobian of a layer.** The gradient of the scalar $J$ with respect to
   $\mathbf{W}^{(2)} \in \mathbb{R}^{h \times q}$ has the shape of
   $\mathbf{W}^{(2)}$. The derivative of the matrix
   $\mathbf{O} = \mathbf{H}\mathbf{W}^{(2)} \in \mathbb{R}^{n \times q}$
   with respect to $\mathbf{W}^{(2)}$ is a larger object, with one entry
   $\partial o_{ij} / \partial w_{rt}$ for every combination of the four
   indices.
    1. How many entries does $\partial \mathbf{O} / \partial \mathbf{W}^{(2)}$
       have, and how many of them can be nonzero?
    1. Evaluate the chain rule
       $\partial J / \partial w_{rt} = \sum_{i, j} (\partial J / \partial o_{ij})(\partial o_{ij} / \partial w_{rt})$
       to recover $\mathbf{H}^\top \partial J / \partial \mathbf{O}$.
    1. How much memory would $\partial \mathbf{O} / \partial \mathbf{W}^{(2)}$
       take in 32-bit floating point for the MLP of
       that section with $n = 256$? Explain why
       backpropagation never forms it.
1. **Modified activation.** The first hidden unit of the network in
   the equation receives the pre-activation $z_1 = -1$, so
   the ReLU derivative $\phi'(z_1) = 0$ blocks the gradient of every weight
   into this unit. Replace the activation of this unit alone by the leaky
   variant $\phi_1(z) = \max(0, z) + 0.1 \min(0, z)$; the second unit keeps
   its ReLU.
    1. Write the backward rule for $\partial J / \partial \mathbf{Z}$ when
       hidden unit $j$ has its own activation $\phi_j$.
    1. Redo the forward and backward passes, and give
       $\partial L / \partial \mathbf{W}^{(2)}$,
       $\partial L / \partial \mathbf{Z}$, and
       $\partial L / \partial \mathbf{W}^{(1)}$. Which of these gradients
       differ from the all-ReLU case even though the rules that produce
       them are unchanged, and why?

    *Adapted from Michael Nielsen,
    [Neural Networks and Deep Learning](http://neuralnetworksanddeeplearning.com/chap2.html),
    Chapter 2, and Simon Prince,
    [Understanding Deep Learning](https://udlbook.github.io/udlbook/),
    Problem 7.10.*
1. **Memory of a deeper network.** Consider an MLP with $d$ inputs, $q$
   outputs, and $K$ layers, of which the $K - 1$ hidden ones are ReLU layers
   of width $h$, trained on minibatches of $n$ examples.
    1. Using the step for $K$ layers in the text, list the values that the
       backward pass reads, and count them. Count the parameters.
    1. Prediction can discard each value as soon as the next layer has used
       it. Count the largest number of values that prediction holds at any
       moment.
    1. For $d = 784$, $h = 256$, and $q = 10$, find the smallest minibatch
       size at which the stored values outnumber the parameters, for
       $K = 2$ and for $K = 11$.
1. [code] **Gradient checking.** Before automatic differentiation, a
   hand-written backward pass was checked against finite differences.
    1. Pick one entry $w$ of $\mathbf{W}^{(1)}$ and compare its gradient from
       `mlp_backward` with the central difference
       $\bigl(J(w + \epsilon) - J(w - \epsilon)\bigr) / (2\epsilon)$ for
       $\epsilon = 10^{-1}, 10^{-2}, \ldots, 10^{-7}$, once in 32-bit and once
       in 64-bit floating point. Plot the relative error against $\epsilon$
       on logarithmic axes.
    1. Explain the shape of the curves. Which error decreases with
       $\epsilon$, which increases, and where do they balance in each
       precision?
    1. The check can fail at a correct gradient when a pre-activation lies
       within $\epsilon$ of zero. Explain why, and how to avoid it.
1. **Forward mode.** Backpropagation seeds $\partial J / \partial J = 1$ at
   the output and sweeps the graph backward. Forward-mode differentiation
   instead seeds $\partial w / \partial w = 1$ at one chosen scalar parameter
   $w$ and sweeps forward: each node computes the derivative of its value
   with respect to $w$ from the derivatives of its inputs.
    1. Use forward mode to recompute $\partial L / \partial \mathbf{W}^{(1)}$
       for the network in the equation, one sweep per entry
       of $\mathbf{W}^{(1)}$.
    1. How many sweeps does each mode need to obtain all six weight
       gradients of this network? Which products does forward mode compute
       again in every sweep that reverse mode computes only once?

    *Adapted from Simon Prince,
    [Understanding Deep Learning](https://udlbook.github.io/udlbook/),
    Problems 7.12 and 7.13.*
1. **Hessian of a ReLU network.** ● ReLU has $\phi''(z) = 0$ for
   $z \neq 0$. For the network in the equation, derive the
   $4 \times 4$ Hessian of $L$ with respect to the entries of
   $\mathbf{W}^{(1)}$, show that it has rank at most one, and evaluate it at
   the given values.
1. [extended] **Autograd engine.** ● Build a miniature automatic
   differentiation engine and use it to recompute the gradients of the
   network in the equation.
    1. Write a scalar `Value` class that records, for each result, its
       inputs and the operation that produced it (the computational graph),
       supporting `+`, `*`, and `relu`. (*Hint:* implement `__add__` and
       `__mul__` so that each returns a new `Value` holding references to
       its parents and a small function that propagates the gradient one
       step.)
    1. Implement `backward()`: sort the graph topologically, seed the
       output's gradient with $1$, and visit the nodes in reverse order,
       letting each node pass its gradient to its parents. Make sure that
       gradients *accumulate* (`+=`, not `=`) when a value is used more than
       once.
    1. Recompute the graph of the figure and the
       worked example the equation with your engine (unroll
       the matrix products into scalars), and compare the gradients with
       those derived in this section.
    1. Consider the chain in which a scalar input $x$ produces
       $v_i = f_i(x)$ for $i = 1, 2, 3$, the intermediate value
       $m = v_1 + v_2 + v_3$, the values $p_j = g_j(m)$ for $j = 1, 2, 3$,
       and the loss $L = p_1 + p_2 + p_3$. Write $\alpha_i = f_i'(x)$ and
       $\beta_j = g_j'(m)$. The chain rule as a sum over paths enumerates
       $3 \times 3 = 9$ paths from $x$ to $L$, each contributing the product
       $\alpha_i \beta_j$. Show that your engine instead computes
       $(\alpha_1 + \alpha_2 + \alpha_3)(\beta_1 + \beta_2 + \beta_3)$ by
       accumulating the gradient at $m$ once, and explain why this sharing
       keeps backpropagation affordable in deep networks.

    *Adapted from Andrej Karpathy,
    [micrograd](https://github.com/karpathy/micrograd),
    `micrograd/engine.py`.*

[Discussions](https://d2l.discourse.group/t/102)